# P3-01 · D-FINE T4 feasibility (512 vs 640, AMP)

Measures, on a **synthetic** COCO set (no real images, no sealed data), for D-FINE-S at input 512 and 640:
peak GPU memory per batch size at the largest multi-scale training size, the largest batch that fits in
14 GB, and seconds per epoch (training + development evaluation) at one common batch.

**Before you start (Kaggle):** Accelerator = **GPU T4** (T4 x2 is fine, only GPU 0 is used), Internet = **On**,
and **+ Add Input** → your private dataset with `p3_code_bundle.zip` (made by `python -m p3_ops.kaggle_bundle`).

Run with **Run All**. The **last cell** prints one JSON block: copy all of it and paste it back to Claude Code.

In [ ]:
# Cell 1: settings (normally nothing to change)

import json, os, subprocess, sys
from pathlib import Path


def sh(cmd, **kw):
    """Run a command and stream its output into the notebook (plain subprocess output is not shown)."""
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace", **kw)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")

SMOKE = os.environ.get("P3_NOTEBOOK_SMOKE") == "1"   # set only by the local CPU smoke test, never on Kaggle
MODEL = "s"                    # the variant in src/p3_det/configs/dfine_*.yaml
N_TRAIN, N_DEV = 5940, 660     # methodology 1.2 training / development image counts (approximate until P1's C01)
SEED = 0
DFINE_COMMIT = "956d1709314c2c6a4df6f34de232054578a7449f"
REPO_URL = ""                  # leave empty: code comes from the Kaggle dataset; set once a team repo exists
if not SMOKE:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("smoke test" if SMOKE else "Kaggle run", "| model", MODEL, "| seed", SEED)

In [ ]:
# Cell 2: get the P3 code and D-FINE at the pinned commit
import glob, shutil

if SMOKE:
    REPO = Path(os.environ["P3_REPO"])
elif REPO_URL:
    REPO = Path("/kaggle/working/cv_proj")
    if not REPO.exists():
        sh(["git", "clone", "-q", REPO_URL, REPO])
else:
    REPO = Path("/kaggle/working/cv_proj")
    hits = glob.glob("/kaggle/input/**/notebooks/p3/p3_01_dfine_feasibility.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/p3_code_bundle.zip", recursive=True)
    if not REPO.exists() and hits:                      # Kaggle unpacked the zip
        shutil.copytree(Path(hits[0]).parents[2], REPO)
    elif not REPO.exists() and zips:                    # still a zip
        shutil.unpack_archive(zips[0], REPO)
    elif not REPO.exists():
        raise SystemExit("P3 code not found: add your private dataset with p3_code_bundle.zip (+ Add Input)")
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
DFINE = REPO / "third_party" / "D-FINE"
if not (DFINE / "train.py").exists():
    sh(["git", "clone", "-q", "https://github.com/Peterande/D-FINE", DFINE])
    sh(["git", "-C", DFINE, "checkout", "-q", DFINE_COMMIT])
head = subprocess.run(["git", "-C", str(DFINE), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == DFINE_COMMIT, f"D-FINE is at {head}, expected {DFINE_COMMIT}"
print("P3 code:", REPO, "| D-FINE:", head)

In [ ]:
# Cell 3: install P3 extras (Kaggle already has CUDA torch; torch is NOT reinstalled)
if not SMOKE:
    sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements/p3.txt"])
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
if not SMOKE and not torch.cuda.is_available():
    raise SystemExit("No GPU: set Accelerator = GPU T4 in the session options and restart")

In [ ]:
# Cell 4: the measurement (about 30-60 min on a T4 for D-FINE-S at 512 and 640)
OUT = Path(os.environ["P3_SMOKE_OUT"]) if SMOKE else Path("/kaggle/working/p3_feasibility")
if SMOKE:
    args = ["--smoke", "--work-dir", OUT]
else:
    args = ["--models", MODEL, "--sizes", 512, 640, "--workers", 4, "--seed", SEED, "--out", OUT]
sh([sys.executable, "notebooks/p3/p3_01_dfine_feasibility.py", *args, "--n-train", N_TRAIN, "--n-dev", N_DEV])

**Last cell.** Copy everything it prints (one JSON block, from `{` to `}`) and paste it back to Claude Code.

In [ ]:
# Cell 5: the ONE block to paste back
from p3_det.fill_training_config import paste_block
found = sorted(OUT.rglob("dfine_t4_*.json")) or sorted(OUT.rglob("smoke_cpu.json"))
print(json.dumps(paste_block(json.loads(found[-1].read_text()), MODEL), indent=1))